# CASIA Convolutional VAE Training — Colab T4

Select **Runtime → Change runtime type → T4 GPU**, then run all cells. This notebook trains only the VAE.

In [1]:
%pip install -q kagglehub Pillow matplotlib numpy
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime before training."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)

GPU: Tesla T4


In [2]:
import os, sys, subprocess
from pathlib import Path
REPO_URL = "https://github.com/chetanraje27/Digital-Evidence-GenAI.git"
PROJECT_ROOT = Path("/content/Digital-Evidence-GenAI")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [3]:
import kagglehub
expected = PROJECT_ROOT / "data/raw/CASIA2"
if not (expected / "Au").is_dir():
    downloaded = Path(kagglehub.dataset_download("divg07/casia-20-image-tampering-detection-dataset")).resolve()
    candidates = [downloaded] + list(downloaded.rglob("CASIA2"))
    actual = next(p for p in candidates if (p/"Au").is_dir() and (p/"Tp").is_dir())
    expected.parent.mkdir(parents=True, exist_ok=True)
    if expected.is_symlink(): expected.unlink()
    expected.symlink_to(actual, target_is_directory=True)
assert (expected/"Au/Au_ani_00001.jpg").is_file()
print("CASIA and portable split paths: READY")

Using Colab cache for faster access to the 'casia-20-image-tampering-detection-dataset' dataset.
CASIA and portable split paths: READY


In [4]:
from argparse import Namespace
from train_vae import train
args = Namespace(
    splits_dir=PROJECT_ROOT/"data/splits",
    checkpoint_path=PROJECT_ROOT/"checkpoints/best_vae.pth",
    history_path=PROJECT_ROOT/"results/vae_training_history.csv",
    total_curve_path=PROJECT_ROOT/"outputs/vae/vae_total_loss_curve.png",
    component_curve_path=PROJECT_ROOT/"outputs/vae/vae_reconstruction_kl_curve.png",
    reconstruction_grid_path=PROJECT_ROOT/"outputs/vae/vae_reconstruction_grid.png",
    random_samples_path=PROJECT_ROOT/"outputs/vae/vae_random_samples.png",
    image_size=128, batch_size=32, num_workers=0, latent_dim=128,
    learning_rate=0.0005, max_epochs=30, patience=4, beta=0.001, seed=42,
    smoke_test=False, smoke_batches=2,
)
summary = train(args)

Epoch 01/30 | train_total=0.05680367 | val_total=0.05061452 | recon=0.04278143 | kl=7.833098 | seconds=122.9
Epoch 02/30 | train_total=0.04955200 | val_total=0.04942099 | recon=0.04377182 | kl=5.649170 | seconds=51.2
Epoch 03/30 | train_total=0.04870773 | val_total=0.04805534 | recon=0.04137347 | kl=6.681868 | seconds=49.7
Epoch 04/30 | train_total=0.04628413 | val_total=0.04533762 | recon=0.03915352 | kl=6.184100 | seconds=49.4
Epoch 05/30 | train_total=0.04454982 | val_total=0.04406497 | recon=0.03738118 | kl=6.683791 | seconds=49.5
Epoch 06/30 | train_total=0.04382549 | val_total=0.04368656 | recon=0.03702449 | kl=6.662061 | seconds=49.4
Epoch 07/30 | train_total=0.04348546 | val_total=0.04365354 | recon=0.03702737 | kl=6.626164 | seconds=49.7
Epoch 08/30 | train_total=0.04333169 | val_total=0.04384038 | recon=0.03688372 | kl=6.956668 | seconds=49.4
Epoch 09/30 | train_total=0.04291720 | val_total=0.04283416 | recon=0.03556423 | kl=7.269935 | seconds=49.5
Epoch 10/30 | train_total=0

In [5]:
print("GPU:", summary["gpu"])
print("Epochs completed:", summary["epochs_completed"])
print("Best epoch:", summary["best_epoch"])
print("First train total loss:", summary["first_train_total_loss"])
print("Final train total loss:", summary["final_train_total_loss"])
print("Best validation total loss:", summary["best_validation_total_loss"])
print("Best validation reconstruction loss:", summary["best_validation_reconstruction_loss"])
print("Best validation KL loss:", summary["best_validation_kl_loss"])
print("Training time:", summary["training_time_seconds"])
print("Checkpoint path:", summary["checkpoint_path"])
print("Errors/warnings: None")

GPU: Tesla T4
Epochs completed: 30
Best epoch: 30
First train total loss: 0.05680366782798502
Final train total loss: 0.04132097358746815
Best validation total loss: 0.04149372029821162
Best validation reconstruction loss: 0.0342758439055961
Best validation KL loss: 7.217876028056881
Training time: 1554.426332062
Checkpoint path: /content/Digital-Evidence-GenAI/checkpoints/best_vae.pth
Errors/warnings: None
